<a href="https://colab.research.google.com/github/DarkWorldCoder/LearningAI/blob/master/Untitled6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#ATIS Dataset

In [ ]:
file_share_link = "https://drive.google.com/file/d/1N4SnfUQPUMQEvaaBn6F7iG6b-TcUUgtT/view?usp=share_link"

file_id = file_share_link[file_share_link.find("d/")+2 : file_share_link.find("/v")]
print(file_id)
!gdown "$file_id"

In [ ]:
!unzip atis.zip

In [ ]:
!rm atis.zip


In [ ]:
import os
import numpy as np
import random
import pandas as pd
import tensorflow as tf

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.layers import Dense,Input,GlobalMaxPool1D,Conv1D,MaxPooling1D,Embedding,LSTM
from tensorflow.keras.models import Model,Sequential
from tensorflow.keras.initializers import Constant

from sklearn.preprocessing import LabelEncoder

In [ ]:
train_df = pd.read_csv("atis_intents_train.csv",header=None)
train_df.columns = ["intents","texts"]
train_df.head()

,intents,texts
0,atis_flight,i want to fly from boston at 838 am and arriv...
1,atis_flight,what flights are available from pittsburgh to...
2,atis_flight_time,what is the arrival time in san francisco for...
3,atis_airfare,cheapest airfare from tacoma to orlando
4,atis_airfare,round trip fares from pittsburgh to philadelp...


In [ ]:
test_df = pd.read_csv("atis_intents_test.csv",header=None)
test_df.columns = ["intents","texts"]
test_df.head()

,intents,texts
0,atis_flight,i would like to find a flight from charlotte ...
1,atis_airfare,on april first i need a ticket from tacoma to...
2,atis_flight,on april first i need a flight going from pho...
3,atis_flight,i would like a flight traveling one way from ...
4,atis_flight,i would like a flight from orlando to salt la...


In [ ]:
MAX_SEQUENCE_LENGTH = 300
MAX_NUM_WORDS = 20000
EMBEDDING_DIM = 100
VALIDATION_SPLIT = 0.3

In [ ]:
tokenizer= Tokenizer(num_words = MAX_NUM_WORDS)
tokenizer.fit_on_texts(train_df["texts"])

train_sequences = tokenizer.texts_to_sequences(train_df["texts"])
test_sequences = tokenizer.texts_to_sequences(test_df["texts"])
word_index = tokenizer.word_index


In [ ]:
set(train_df["intents"])

{'atis_abbreviation',
 'atis_aircraft',
 'atis_airfare',
 'atis_airline',
 'atis_flight',
 'atis_flight_time',
 'atis_ground_service',
 'atis_quantity'}

In [ ]:
le = LabelEncoder()
le.fit(train_df["intents"])
train_df["intents"] = le.transform(train_df["intents"])
test_df["intents"] = le.transform(test_df["intents"])

In [ ]:
max([len(l) for l in train_sequences])

46

In [ ]:
train_valid_data = pad_sequences(train_sequences,maxlen=MAX_SEQUENCE_LENGTH)
test_data = pad_sequences(test_sequences,maxlen=MAX_SEQUENCE_LENGTH)

train_valid_labels = to_categorical(train_df["intents"])
test_labels = to_categorical(test_df["intents"])


In [ ]:
from sklearn.model_selection import train_test_split
X_train,X_val,y_train,y_val = train_test_split(train_valid_data,train_valid_labels,test_size=VALIDATION_SPLIT,random_state=42)


Embeddings


In [ ]:
file_share_link = "https://drive.google.com/file/d/1qqTDo8h4WDcNBEFnOH-LEPhra0zjyoXj/view?usp=share_link"

file_id = file_share_link[file_share_link.find("d/")+2 : file_share_link.find("/v")]
import gdown
gdown.download(f"https://drive.google.com/uc?export=download&confirm=pbef&id={file_id}")


Downloading...
From: https://drive.google.com/uc?export=download&confirm=pbef&id=1qqTDo8h4WDcNBEFnOH-LEPhra0zjyoXj
To: /content/glove.6B.100d.txt.zip
100%|██████████| 138M/138M [00:01<00:00, 124MB/s]


'glove.6B.100d.txt.zip'

In [ ]:
!unzip glove.6B.100d.txt.zip


Archive:  glove.6B.100d.txt.zip
replace glove.6B.100d.txt? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [ ]:
GLOVE_DIR = "/content"

embeddings_index = {}
with open(os.path.join(GLOVE_DIR,"glove.6B.100d.txt"),encoding="utf-8") as f:
  for line in f:
    values = line.split()
    word = values[0]
    coefs = np.asarray(values[1:],dtype="float32")
    embeddings_index[word] = coefs
num_words = min(MAX_NUM_WORDS,len(word_index)) +1
embedding_matrix = np.zeros((num_words,EMBEDDING_DIM))
for word, i in word_index.items():
  if i > MAX_NUM_WORDS:
    continue
  embedding_vector = embeddings_index.get(word)
  if embedding_vector is not None:
    embedding_matrix[i] = embedding_vector

embedding_layer = Embedding(num_words,
                            EMBEDDING_DIM,
                            embeddings_initializer = Constant(embedding_matrix)
                            ,input_length = MAX_SEQUENCE_LENGTH,
                            trainable=False
                            )


In [ ]:
model = Sequential()
model.add(embedding_layer)

model.add(Conv1D(filters=128,kernel_size=5,activation="relu"))
model.add(MaxPooling1D(5))
model.add(Conv1D(filters=128,kernel_size=5,activation="relu"))
model.add(MaxPooling1D(5))

model.add(Conv1D(filters=128,kernel_size=5,activation="relu"))
model.add(MaxPooling1D(5))

model.add(Dense(128,activation="relu"))
model.add(Dense(8,activation="softmax"))

In [ ]:
model.compile(loss="categorical_crossentropy",
              optimizer="rmsprop",
              metrics=["acc"]
              )
model.summary()

In [ ]:
model.fit(X_train,y_train,batch_size=128,epochs=5,validation_data=(X_val,y_val))